<!--
---
PURPOSE: Align Facemap + SuperAnimal outputs to NWB timebase and build
         unified pose_features.parquet for encoding model.
REQUIRES:
  - outputs/pose/session_{id}_facemap_eye.parquet
  - outputs/pose/session_{id}_facemap_face.parquet
  - outputs/pose/session_{id}_superanimal.h5
  - outputs/video/frame_times_{id}_{camera}.parquet
PRODUCES:
  - outputs/pose/session_{id}_pose_features.parquet
  - outputs/pose/session_{id}_motifs.parquet
WHAT_NEXT: notebooks/08_Neural_Behavior_Fusion_and_Modeling.ipynb
---
-->

# 07 Pose Features — Align Facemap + SuperAnimal to NWB timebase

Converts raw Facemap and SuperAnimal outputs to a unified feature timeseries
on the NWB seconds clock. The output (`pose_features.parquet`) feeds directly
into the NB08 encoding model as additional behavioral covariates.

**Priority of signals for encoding model (Stringer 2019):**
1. Running speed (already in NWB) — fastest to compute
2. Pupil area from Facemap eye tracker
3. Face SVD motion energy PCs (whisking, sniffing, grooming)
4. Body keypoints from SuperAnimal (lowest priority for head-fixed mouse)


## Environment Setup

In [4]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
# Works whether JupyterLab is launched from repo root or from notebooks/
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
SRC  = ROOT / "src"

# put repo + src on sys.path
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


#print("ROOT:", ROOT)
#print("SRC :", SRC, "| exists:", SRC.exists())
#print("sys.path[0:3]:", sys.path[:3])

## Step 1: Align pose outputs to NWB timebase

Facemap outputs frame indices; SuperAnimal outputs frame-indexed HDF5.
We use the camera frame_times (from NB05) to convert frame index → NWB seconds.
If frame_times are not available, we fall back to linear interpolation over
the session duration from the NWB spike time range.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from config import get_config, make_provenance
from io_sessions import load_sessions_csv, get_session_bundle
from io_video import load_frame_times
from timebase import write_parquet_with_timebase

cfg = get_config()
sessions = load_sessions_csv()
SESSION_IDS = [1055240613]

POSE_DIR = cfg.outputs_dir / "pose"
MIN_LIKELIHOOD = 0.6   # SuperAnimal keypoint confidence threshold

def _align_frames_to_nwb(df, session_id, camera, fallback_duration_s=None):
    """Convert frame-index column to NWB seconds using frame_times.
    Falls back to linear interpolation if frame_times not available."""
    try:
        ft = load_frame_times(session_id=session_id, camera=camera)
        if ft is not None and "frame" in ft.columns and "t" in ft.columns:
            df = df.merge(ft[["frame", "t"]], on="frame", how="left", suffixes=("_approx", ""))
            if "t_approx" in df.columns:
                df = df.drop(columns=["t_approx"])
            return df
    except Exception:
        pass

    # Fallback: linear spacing over session duration
    if fallback_duration_s is not None and len(df) > 0:
        df["t"] = np.linspace(0, fallback_duration_s, len(df))
        print(f"  WARNING: Using linear t fallback — run NB05 first for accurate frame times")
    return df

pose_features_all = {}  # sid -> merged features DataFrame

for sid in SESSION_IDS:
    print(f"\n[{sid}]")
    bundle = get_session_bundle(sid, sessions)
    parts = []

    # Estimate session duration from spike times for t fallback
    spike_path = cfg.outputs_dir / "neural" / f"session_{sid}_spike_times.npz"
    fallback_dur = None
    if spike_path.exists():
        import io_nwb
        spikes = dict(io_nwb.load_spike_times_npz(spike_path))
        if spikes:
            all_t = np.concatenate(list(spikes.values()))
            fallback_dur = float(all_t.max() - all_t.min())

    # ── Facemap eye (pupil area / position / blink) ──────────────────────
    eye_path = POSE_DIR / f"session_{sid}_facemap_eye.parquet"
    if eye_path.exists():
        df_eye = pd.read_parquet(eye_path)
        df_eye = _align_frames_to_nwb(df_eye, sid, "eye", fallback_dur)
        # Filter blink frames
        if "blink" in df_eye.columns:
            n_blinks = df_eye["blink"].sum()
            df_eye.loc[df_eye["blink"], ["pupil_area", "pupil_x", "pupil_y"]] = np.nan
            df_eye["pupil_area"] = df_eye["pupil_area"].interpolate("linear")
            print(f"  Facemap eye: {len(df_eye)} frames, {n_blinks} blink frames interpolated")
        keep_cols = [c for c in ["t", "pupil_area", "pupil_x", "pupil_y"] if c in df_eye.columns]
        parts.append(df_eye[keep_cols].rename(columns={"pupil_area": "pupil"}))
    else:
        print(f"  Facemap eye: not found ({eye_path.name})")

    # ── Facemap face SVD ─────────────────────────────────────────────────
    face_path = POSE_DIR / f"session_{sid}_facemap_face.parquet"
    if face_path.exists():
        df_face = pd.read_parquet(face_path)
        df_face = _align_frames_to_nwb(df_face, sid, "face", fallback_dur)
        svd_cols = [c for c in df_face.columns if c.startswith("face_svd_")]
        print(f"  Facemap face SVD: {len(df_face)} frames, {len(svd_cols)} components")
        parts.append(df_face[["t"] + svd_cols])
    else:
        print(f"  Facemap face SVD: not found ({face_path.name})")

    # ── SuperAnimal keypoints ─────────────────────────────────────────────
    sa_path = POSE_DIR / f"session_{sid}_superanimal.h5"
    if sa_path.exists():
        try:
            df_sa = pd.read_hdf(sa_path, key="df_with_missing")
            df_sa.columns = ["_".join(c).strip("_") for c in df_sa.columns]
            df_sa = df_sa.reset_index(drop=True)
            df_sa.insert(0, "frame", np.arange(len(df_sa)))
            df_sa = _align_frames_to_nwb(df_sa, sid, "side", fallback_dur)

            # Filter low-likelihood keypoints
            likelihood_cols = [c for c in df_sa.columns if c.endswith("_likelihood")]
            for lc in likelihood_cols:
                kp = lc.replace("_likelihood", "")
                for coord in ["_x", "_y"]:
                    col = kp + coord
                    if col in df_sa.columns:
                        df_sa.loc[df_sa[lc] < MIN_LIKELIHOOD, col] = np.nan

            # Compute speed per keypoint as aggregate motion signal
            x_cols = [c for c in df_sa.columns if c.endswith("_x") and not c.endswith("likelihood_x")]
            if x_cols and "t" in df_sa.columns:
                dt = df_sa["t"].diff().median()
                speeds = []
                for xc in x_cols:
                    yc = xc.replace("_x", "_y")
                    if yc in df_sa.columns:
                        dx = df_sa[xc].diff()
                        dy = df_sa[yc].diff()
                        speeds.append(np.sqrt(dx**2 + dy**2) / (dt + 1e-9))
                if speeds:
                    df_sa["body_speed"] = np.nanmean(np.stack(speeds, axis=1), axis=1)

            keep = ["t", "body_speed"] + [c for c in df_sa.columns
                    if c.endswith("_x") or c.endswith("_y")
                    if not c.endswith("likelihood_x") and c != "t"]
            keep = [c for c in keep if c in df_sa.columns]
            parts.append(df_sa[keep])
            print(f"  SuperAnimal: {len(df_sa)} frames, {len(x_cols)} keypoints")
        except Exception as e:
            print(f"  SuperAnimal: ERROR loading {sa_path.name}: {e}")
    else:
        print(f"  SuperAnimal: not found ({sa_path.name})")

    if not parts:
        print(f"  No pose data found — skipping feature merge")
        continue

    # ── Merge all parts on NWB seconds timebase ───────────────────────────
    from functools import reduce
    merged = reduce(
        lambda a, b: pd.merge_asof(
            a.sort_values("t"), b.sort_values("t"),
            on="t", direction="nearest", tolerance=0.1,
        ),
        parts,
    )
    merged = merged.dropna(subset=["t"])
    pose_features_all[sid] = merged

    out_path = POSE_DIR / f"session_{sid}_pose_features.parquet"
    write_parquet_with_timebase(
        merged, out_path,
        provenance=make_provenance(sid, "facemap+superanimal"),
        required_columns=["t"],
    )
    print(f"  Merged pose features: {merged.shape} → {out_path.name}")
    print(f"  Columns: {[c for c in merged.columns if c != 't'][:10]} ...")


In [ ]:
# --- Optional: Run automated SLEAP inference on cached videos ---
# This requires a trained SLEAP model. With your ~150 labeled frames,
# you can train one via: sleap-train <config.json> <labels.slp>
# Then place the model in pose_projects/ or data/sleap_models/

if RUN_INFERENCE:
    models = discover_sleap_models()
    if models:
        print(f"Found {len(models)} SLEAP model(s):")
        for m in models:
            print(f"  - {m['path']} ({m.get('type', 'unknown')})")

        # Only run on sessions that don't already have predictions
        missing_sids = [
            sid for sid in SESSION_IDS
            if not (cfg.outputs_dir / "pose" / f"session_{sid}_pose_predictions.parquet").exists()
        ]
        if missing_sids:
            print(f"\nRunning inference on {len(missing_sids)} sessions without predictions...")
            inference_results = run_batch_inference(
                session_ids=missing_sids,
                cameras=[CAMERA],
                model_paths=[models[0]["path"]],
                skip_existing=True,
            )
            print(inference_results[["session_id", "camera", "n_frames", "status"]])
        else:
            print("\nAll sessions already have predictions.")
    else:
        print("No SLEAP models found. To enable automated inference:")
        print("  1. Train a model: sleap-train <config.json> <your_labels.slp>")
        print("  2. Place the model directory in pose_projects/ or data/sleap_models/")
        print("  3. Re-run this cell")

## Step 2: Feature extraction and motif discovery

For each session with predictions, we:
1. **Filter by confidence** — drop low-quality keypoint detections
2. **Extract rich features** — velocity, acceleration, body length, head angle, stillness
3. **Discover motifs** — cluster behavioral features into discrete states (k-means or HMM)

In [ ]:
from motifs import motifs_kmeans, motifs_hmm
from viz import plot_motif_transition

# --- Configuration ---
N_MOTIF_CLUSTERS = 8         # number of behavioral motifs
MOTIF_METHOD = "kmeans"      # "kmeans" or "hmm"

for sid in SESSION_IDS:
    pose_path = cfg.outputs_dir / "pose" / f"session_{sid}_pose_predictions.parquet"

    if not pose_path.exists():
        print(f"\n[session {sid}] SKIP - no predictions found")
        continue

    print(f"\n{'='*60}")
    print(f"[session {sid}] Processing pose predictions")
    print(f"{'='*60}")

    # Load predictions
    pose_df = pd.read_parquet(pose_path)
    if "camera" in pose_df.columns:
        pose_df = pose_df[pose_df["camera"] == CAMERA].reset_index(drop=True)
    print(f"  Raw predictions: {len(pose_df)} rows, {len(pose_df.columns)} columns")

    # Show available keypoints
    kp_cols = [c for c in pose_df.columns if c.endswith("_x")]
    keypoint_names = [c[:-2] for c in kp_cols]
    print(f"  Keypoints found: {keypoint_names}")

    # Show confidence stats (if available)
    score_cols = [c for c in pose_df.columns if c.endswith("_score")]
    if score_cols:
        mean_scores = pose_df[score_cols].mean()
        print(f"  Mean confidence per keypoint:")
        for col, score in mean_scores.items():
            print(f"    {col}: {score:.3f}")

    # Filter by confidence
    if CONFIDENCE_THRESHOLD > 0:
        before = pose_df[kp_cols].notna().sum().sum()
        pose_df = filter_by_confidence(pose_df, CONFIDENCE_THRESHOLD, method="nan")
        after = pose_df[kp_cols].notna().sum().sum()
        pct_kept = 100 * after / before if before > 0 else 0
        print(f"  Confidence filter: kept {pct_kept:.1f}% of keypoint detections")

    # Extract rich features
    features = derive_pose_features(pose_df, confidence_threshold=0)  # already filtered above
    if features is None or features.empty:
        print(f"  WARNING: No features extracted for session {sid}")
        continue

    print(f"  Features extracted: {list(features.columns)}")
    print(f"  Feature shape: {features.shape}")

    # Save features
    features_path = cfg.outputs_dir / "pose" / f"session_{sid}_pose_features.parquet"
    write_parquet_with_timebase(
        features, features_path,
        provenance=make_provenance(sid, "nwb"),
        required_columns=["t"],
    )

    # Discover motifs
    if MOTIF_METHOD == "hmm":
        motifs_df = motifs_hmm(features, n_states=N_MOTIF_CLUSTERS)
    else:
        motifs_df = motifs_kmeans(features, n_clusters=N_MOTIF_CLUSTERS)

    motifs_path = cfg.outputs_dir / "pose" / f"session_{sid}_motifs.parquet"
    write_parquet_with_timebase(
        motifs_df, motifs_path,
        provenance=make_provenance(sid, "nwb"),
        required_columns=["t", "motif_id"],
    )
    print(f"  Motifs: {motifs_df['motif_id'].nunique()} clusters, {len(motifs_df)} time bins")

    # Visualize motif transitions
    plot_motif_transition(motifs_df)

    # Feature summary statistics
    print(f"\n  Feature summary:")
    for col in ["pose_speed", "body_length", "head_angle", "is_still"]:
        if col in features.columns:
            vals = features[col].dropna()
            print(f"    {col}: mean={vals.mean():.3f}, std={vals.std():.3f}, range=[{vals.min():.3f}, {vals.max():.3f}]")

    print(f"  Saved: {features_path}")
    print(f"  Saved: {motifs_path}")

In [ ]:
# --- Active learning: suggest frames to label next ---
# If you have .slp prediction files, this tells you which frames
# the model is LEAST confident about -- labeling those gives you
# the biggest improvement per frame labeled.

print("Active learning suggestions:")
print("=" * 40)

for sid in SESSION_IDS:
    slp_candidates = list((cfg.outputs_dir / "pose" / "predictions").glob(f"*{sid}*.slp"))
    if not slp_candidates:
        print(f"[session {sid}] No .slp predictions found (skip active learning)")
        continue

    try:
        suggestions = suggest_frames_to_label(
            slp_candidates[0],
            n_suggestions=20,
            strategy="spread",
        )
        print(f"[session {sid}] Suggested frames to label next (lowest confidence, spread across session):")
        print(f"  Frame indices: {suggestions.tolist()}")
        print(f"  To label these, open SLEAP GUI with:")
        print(f"    sleap-label {slp_candidates[0]}")
    except Exception as e:
        print(f"[session {sid}] Active learning error: {e}")

print("\nNotebook 07 complete. Run Notebook 08 for neural-behavior fusion.")